# Thermodynamic inference using the three slowest cooling rates

This notebook infers equilibrium microstate probabilities and microstate entropies using only the three slowest observed cooling trajectories. States with hot-equilibrium probability greater than 0.01 are retained individually; the remaining states are combined into a single `other` state.

The paper calculation uses 10,000 deterministic seeds. The feasible probability and entropy samples are saved to `feasible_p.csv` and `feasible_s.csv` for later use.

In [ ]:
import time

import numpy as np
import pandas as pd
from scipy.optimize import NonlinearConstraint, differential_evolution, minimize


## Input and state-space construction

Load the cooling summary, separate the seven cooling conditions and two equilibrium distributions, reduce the original state space to the highest-population hot states plus `other`, and retain only the three slowest cooling conditions for inference.

In [ ]:
# Input and data-handling configuration
INPUT_FILE = "cooling_pairs_summary_k50_t250_lag_400.csv"
HOT_EQM_THRESHOLD = 0.01
tau_list = np.array([800, 1800, 8000])

dataframe = pd.read_csv(INPUT_FILE)

states = np.array(dataframe['state'])

state_number = max(states) + 1


# Cooled trajectories over all original states
cooled_6e9_full = dataframe['mean'].to_numpy()[0:state_number].astype(float)
cooled_2e9_full = dataframe['mean'].to_numpy()[state_number:2*state_number].astype(float)
cooled_6e8_full = dataframe['mean'].to_numpy()[2*state_number:3*state_number].astype(float)
cooled_2e8_full = dataframe['mean'].to_numpy()[3*state_number:4*state_number].astype(float)
cooled_6e7_full = dataframe['mean'].to_numpy()[4*state_number:5*state_number].astype(float)
cooled_2e7_full = dataframe['mean'].to_numpy()[5*state_number:6*state_number].astype(float)
cooled_6e6_full = dataframe['mean'].to_numpy()[6*state_number:7*state_number].astype(float)

cooled_6e9_low_full = dataframe['ci_lower_95'].to_numpy()[0:1*state_number].astype(float)
cooled_2e9_low_full = dataframe['ci_lower_95'].to_numpy()[1*state_number:2*state_number].astype(float)
cooled_6e8_low_full = dataframe['ci_lower_95'].to_numpy()[2*state_number:3*state_number].astype(float)
cooled_2e8_low_full = dataframe['ci_lower_95'].to_numpy()[3*state_number:4*state_number].astype(float)
cooled_6e7_low_full = dataframe['ci_lower_95'].to_numpy()[4*state_number:5*state_number].astype(float)
cooled_2e7_low_full = dataframe['ci_lower_95'].to_numpy()[5*state_number:6*state_number].astype(float)
cooled_6e6_low_full = dataframe['ci_lower_95'].to_numpy()[6*state_number:7*state_number].astype(float)

cooled_6e9_high_full = dataframe['ci_upper_95'].to_numpy()[0*state_number:1*state_number].astype(float)
cooled_2e9_high_full = dataframe['ci_upper_95'].to_numpy()[1*state_number:2*state_number].astype(float)
cooled_6e8_high_full = dataframe['ci_upper_95'].to_numpy()[2*state_number:3*state_number].astype(float)
cooled_2e8_high_full = dataframe['ci_upper_95'].to_numpy()[3*state_number:4*state_number].astype(float)
cooled_6e7_high_full = dataframe['ci_upper_95'].to_numpy()[4*state_number:5*state_number].astype(float)
cooled_2e7_high_full = dataframe['ci_upper_95'].to_numpy()[5*state_number:6*state_number].astype(float)
cooled_6e6_high_full = dataframe['ci_upper_95'].to_numpy()[6*state_number:7*state_number].astype(float)

eqm_data_full = dataframe['mean'].to_numpy()[7*state_number:8*state_number].astype(float)
cold_eqm_full = dataframe['mean'].to_numpy()[8*state_number:9*state_number].astype(float)

# Keep only states that are hot at equilibrium (p > 0.01).
# All other original states are collapsed into a single "other" state.
original_state_numbers = np.arange(state_number, dtype=int)
hot_state_mask = eqm_data_full > HOT_EQM_THRESHOLD
if not np.any(hot_state_mask):
    raise ValueError("No equilibrium-hot states found above the threshold p > 0.01.")

hot_state_original_numbers = original_state_numbers[hot_state_mask]
other_state_original_numbers = original_state_numbers[~hot_state_mask]


def collapse_hot_other(vec, hot_mask, normalize=True):
    vec = np.asarray(vec, dtype=float)
    hot_vals = vec[hot_mask]
    other_val = vec[~hot_mask].sum()
    collapsed = np.concatenate([hot_vals, [other_val]])
    if normalize:
        total = collapsed.sum()
        if total <= 0:
            raise ValueError("Collapsed distribution has non-positive mass.")
        collapsed = collapsed / total
    return collapsed


cooled_6e9 = collapse_hot_other(cooled_6e9_full, hot_state_mask)
cooled_2e9 = collapse_hot_other(cooled_2e9_full, hot_state_mask)
cooled_6e8 = collapse_hot_other(cooled_6e8_full, hot_state_mask)
cooled_2e8 = collapse_hot_other(cooled_2e8_full, hot_state_mask)
cooled_6e7 = collapse_hot_other(cooled_6e7_full, hot_state_mask)
cooled_2e7 = collapse_hot_other(cooled_2e7_full, hot_state_mask)
cooled_6e6 = collapse_hot_other(cooled_6e6_full, hot_state_mask)

cooled_6e9_low = collapse_hot_other(cooled_6e9_low_full, hot_state_mask, normalize=False)
cooled_2e9_low = collapse_hot_other(cooled_2e9_low_full, hot_state_mask, normalize=False)
cooled_6e8_low = collapse_hot_other(cooled_6e8_low_full, hot_state_mask, normalize=False)
cooled_2e8_low = collapse_hot_other(cooled_2e8_low_full, hot_state_mask, normalize=False)
cooled_6e7_low = collapse_hot_other(cooled_6e7_low_full, hot_state_mask, normalize=False)
cooled_2e7_low = collapse_hot_other(cooled_2e7_low_full, hot_state_mask, normalize=False)
cooled_6e6_low = collapse_hot_other(cooled_6e6_low_full, hot_state_mask, normalize=False)

cooled_6e9_high = collapse_hot_other(cooled_6e9_high_full, hot_state_mask, normalize=False)
cooled_2e9_high = collapse_hot_other(cooled_2e9_high_full, hot_state_mask, normalize=False)
cooled_6e8_high = collapse_hot_other(cooled_6e8_high_full, hot_state_mask, normalize=False)
cooled_2e8_high = collapse_hot_other(cooled_2e8_high_full, hot_state_mask, normalize=False)
cooled_6e7_high = collapse_hot_other(cooled_6e7_high_full, hot_state_mask, normalize=False)
cooled_2e7_high = collapse_hot_other(cooled_2e7_high_full, hot_state_mask, normalize=False)
cooled_6e6_high = collapse_hot_other(cooled_6e6_high_full, hot_state_mask, normalize=False)

eqm_data = collapse_hot_other(eqm_data_full, hot_state_mask)
cold_eqm = collapse_hot_other(cold_eqm_full, hot_state_mask)

# Mapping from reduced-state index to original state number(s).
# For hot states this is a single int; for "other" this is a list of original states.
reduced_state_index_to_original = {
    i: int(orig_state) for i, orig_state in enumerate(hot_state_original_numbers)
}
reduced_state_index_to_original[len(hot_state_original_numbers)] = other_state_original_numbers.tolist()

reduced_state_labels = [f"state_{i}" for i in hot_state_original_numbers] + ["other"]

dist_list = [cooled_6e7, cooled_2e7, cooled_6e6]
dist_low_list = [cooled_6e7_low, cooled_2e7_low, cooled_6e6_low]
dist_high_list = [cooled_6e7_high, cooled_2e7_high, cooled_6e6_high]

## Sampling cooled distributions

Approximate a Dirichlet distribution from each observed mean and 95% confidence interval, then draw one cooled distribution per cooling condition for each seeded inference run.

In [ ]:
def dirichlet_from_mean_ci(mean, low, high):
    """
    Approximate Dirichlet parameters from mean and 95% CI for each component.
    mean, low, high are 1D arrays of same length.
    """
    mean = np.asarray(mean, dtype=float)
    low = np.asarray(low, dtype=float)
    high = np.asarray(high, dtype=float)

    # 95% CI ~ mean ± 1.96 * sigma  =>  sigma ≈ (high - low)/(2*1.96)
    sigma = (high - low) / (2 * 1.96)
    var = sigma**2

    eps = 1e-12
    var = np.maximum(var, eps)
    mean = np.clip(mean, eps, 1 - eps)

    # From Var = mean(1-mean)/(alpha0 + 1)  =>  alpha0 ≈ mean(1-mean)/var - 1
    alpha0_est = mean * (1 - mean) / var - 1.0

    # Retain finite, positive component-wise concentration estimates.
    mask = np.isfinite(alpha0_est) & (alpha0_est > 0)
    if not np.any(mask):
        raise ValueError("Could not infer a positive alpha0 from the CIs.")

    alpha0 = np.median(alpha0_est[mask])  # one shared concentration
    alpha = mean * alpha0  # component-wise alphas

    return alpha


def sample_cooled_distributions(mean, low, high, n_samples=1000, random_state=None):
    if isinstance(random_state, np.random.Generator):
        rng = random_state
    else:
        rng = np.random.default_rng(random_state)

    alpha = dirichlet_from_mean_ci(mean, low, high)
    samples = rng.dirichlet(alpha, size=n_samples)  # shape (n_samples, n_states)
    return samples


## Feasible Distribution Generation

Fit each sampled cooling trajectory with differential evolution, followed when needed by a time-limited constrained-polishing step. The batch runner reports statistics over all fits and over the feasible subset.

In [ ]:
def solve_fit_once_safe(dist_list, tau_list, eqm_data,
                        t_hot=277.0, t_cold=230.0,
                        max_gen=1000,
                        seed=0,
                        do_polish=True,
                        polish_maxiter=200,
                        polish_time_limit=20.0,   # seconds; set None to disable time limit
                        polish_skip_if_gap_ge=-1e-3,  # skip polish if DE worst_gap >= this
                        tiny=1e-15,
                        verbose=False,
                        start_p=None):
    """
    One run with DE + (optional) trust-constr polish.
    Uses polish iteration/time limits and skips polishing if already adequate.
    Returns dict with p_star, s_star, worst_gap, feasible, timings, etc.
    """
    # Sort the sampled distributions by cooling time.
    idx        = np.argsort(tau_list)
    tau_sorted = np.asarray(tau_list)[idx]
    dlist      = [np.array(dist_list[i], dtype=float) for i in idx]

    beta = t_hot / t_cold
    N    = int(eqm_data.size)

    # Parameter and constraint helpers.
    def unpack(theta):
        s, logw = np.split(theta, 2)
        w = np.exp(logw - logw.max());  p = w / w.sum()
        return s, p

    def all_gaps(theta):
        s, p = unpack(theta)
        p = np.clip(p, tiny, None); p /= p.sum()
        p_c = np.clip(p**beta * np.exp((1-beta)*s), tiny, None)
        p_c /= p_c.sum()

        e_eq, s_eq = np.dot(p, -np.log(p)+s), np.dot(p, s-np.log(p))
        e_c , s_c  = np.dot(p_c,-np.log(p)+s), np.dot(p_c, s-np.log(p_c))

        gaps, aux_prev, cool_prev, dE_prev = [], np.inf, -np.inf, None
        for p_tau in dlist:
            p_tau = np.clip(p_tau, tiny, None); p_tau /= p_tau.sum()
            e_t, s_t = np.dot(p_tau, -np.log(p)+s), np.dot(p_tau, s-np.log(p_tau))

            sig_cool = -(e_t-e_eq)*beta + (s_t-s_eq)   # Σ_cool
            sig_aux  =  (e_t-e_c )*beta + (s_c-s_t)    # Σ_aux
            dE_cool  =   e_t - e_eq                    # ΔE_cool
            gaps.extend([sig_cool, sig_aux])

            if aux_prev < np.inf:                      # k ≥ 1
                gaps.append(aux_prev - sig_aux)        # Σ_aux mono
                gaps.append(sig_cool - cool_prev)      # Σ_cool mono
            aux_prev, cool_prev, dE_prev = sig_aux, sig_cool, dE_cool
        return np.asarray(gaps)

    def obj(theta):   # 0 ⇔ all gaps ≥ 0
        g = np.minimum(all_gaps(theta), 0.0)
        return np.dot(g, g)

    #[-2,2] allows a full range for the microstate entropies of 4 >log 50.
    bounds = [(-2, 2)]*N + [(-15, 8)]*N
    lower = np.array([b[0] for b in bounds])
    upper = np.array([b[1] for b in bounds])

    # Optional fixed starting distribution.
    theta0 = None
    if start_p is not None:
        p0 = np.asarray(start_p, float)
        if p0.shape != (N,):
            raise ValueError(f"start_p must have shape ({N},), got {p0.shape}")

        # make it a proper prob. vector, avoid zeros
        p0 = np.clip(p0, tiny, None)
        p0 /= p0.sum()

        # choose some default s_0; zeros is a neutral choice within [-2, 2]
        s0 = np.zeros_like(p0)

        # internal parameterization: p ~ softmax(logw)
        # logw = log(p0) will give back p0 (up to normalization) in unpack()
        logw0 = np.log(p0)

        theta0 = np.concatenate([s0, logw0])

        # respect bounds
        theta0 = np.clip(theta0, lower, upper)

    # Differential evolution with a fixed number of generations.
    popsize = 15
    gen = {'k': 0}
    def de_cb(x, _):
        if verbose and gen['k'] % 50 == 0:
            print(f"  [seed {seed:3d}] DE gen {gen['k']:4d}/{max_gen}: "
                  f"worst_gap={-np.sqrt(obj(x)):+.3e}", flush=True)
        gen['k'] += 1

    # Prepare initial population if we have a starting theta
    de_init = 'latinhypercube'
    if theta0 is not None:
        dim = len(theta0)
        M   = popsize * dim   # DE’s population size
        rng = np.random.default_rng(seed)
        init_pop = rng.uniform(lower, upper, size=(M, dim))
        init_pop[0] = theta0   # first member = your fixed starting p (with s0)
        de_init = init_pop

    t0 = time.time()
    coarse = differential_evolution(
        obj, bounds,
        popsize=popsize, maxiter=max_gen, seed=seed,
        strategy='best1bin',
        tol=0.0,
        callback=de_cb,
        polish=False,
        updating='immediate', workers=1,
        init=de_init
    )
    t_de = time.time() - t0

    theta     = coarse.x
    worst_gap = -np.sqrt(obj(theta))

    # Optional constrained polish with iteration and time limits.
    t_polish = 0.0
    if do_polish and (worst_gap < polish_skip_if_gap_ge):
        nlc = NonlinearConstraint(all_gaps, 0.0, np.inf)

        # time-limited callback (trust-constr supports callback(xk))
        start = time.time()
        def tc_cb(xk, state=None):
            nonlocal start
            if polish_time_limit is not None and (time.time() - start) > polish_time_limit:
                return True  # request termination
            return False

        t1 = time.time()
        try:
            tc = minimize(
                obj, theta, method='trust-constr',
                constraints=[nlc], bounds=bounds,
                options={'maxiter': polish_maxiter,
                         'xtol': 1e-10,
                         'gtol': 1e-10,
                         'barrier_tol': 1e-8,
                         'verbose': 0},
                callback=tc_cb
            )
            if tc.success and obj(tc.x) < obj(theta):
                theta = tc.x
            worst_gap = -np.sqrt(obj(theta))
        except Exception:
            pass
        t_polish = time.time() - t1

    s_star, p_star = unpack(theta)
    feasible = (worst_gap >= -1e-10)

    return {
        'p_star': p_star,
        's_star': s_star,   # microstate entropies per state
        'worst_gap': float(worst_gap),
        'feasible': bool(feasible),
        'seed': int(seed),
        't_de': t_de,
        't_polish': t_polish,
    }



def run_batch_fits_safe(dist_list,dist_high_list,dist_low_list, tau_list, eqm_data,
                        n_runs=100,
                        base_seed=0,
                        max_gen=1000,
                        do_polish=True,
                        polish_maxiter=200,
                        polish_time_limit=20.0,
                        polish_skip_if_gap_ge=-1e-3,
                        tiny=1e-15,
                        verbose=True,start_p=None):
    """
    Sequential (safe) batch loop with per-run limits and progress prints.
    Returns summary with mean/std of p* and s* (all runs and feasible-only).
    """
    seeds = [base_seed + i for i in range(n_runs)]
    results = []
    for j, sd in enumerate(seeds):

        # Draw one independent cooled distribution per tau from a single per-run RNG stream.
        run_rng = np.random.default_rng(sd)
        dist_list_temp = []
        for k in range(len(tau_list)):
            dist_list_temp.append(
                sample_cooled_distributions(
                    dist_list[k], dist_low_list[k], dist_high_list[k],
                    n_samples=1, random_state=run_rng
                )[0]
            )
        if verbose:
            print(f"\n=== Run {j+1}/{n_runs} (seed={sd}) ===", flush=True)
        res = solve_fit_once_safe(
            dist_list_temp, tau_list, eqm_data,
            max_gen=max_gen, seed=sd, do_polish=do_polish,
            polish_maxiter=polish_maxiter,
            polish_time_limit=polish_time_limit,
            polish_skip_if_gap_ge=polish_skip_if_gap_ge,
            tiny=tiny,
            verbose=verbose,start_p=start_p
        )
        if verbose:
            print(f"  seed={sd} worst_gap={res['worst_gap']:+.3e} "
                  f"[t_de={res['t_de']:.1f}s, t_polish={res['t_polish']:.1f}s]  "
                  f"feasible={res['feasible']}", flush=True)
        results.append(res)

    # Stack fitted probabilities and entropies and compute summary statistics.
    P = np.vstack([r['p_star'] for r in results])
    S = np.vstack([r['s_star'] for r in results])

    mean_p_all = P.mean(axis=0)
    std_p_all  = P.std(axis=0, ddof=1) if P.shape[0] > 1 else np.zeros(P.shape[1])

    mean_s_all = S.mean(axis=0)
    std_s_all  = S.std(axis=0, ddof=1) if S.shape[0] > 1 else np.zeros(S.shape[1])

    feas_mask = np.array([r['feasible'] for r in results], dtype=bool)
    if feas_mask.any():
        P_feas = P[feas_mask]
        S_feas = S[feas_mask]
        mean_p_feas = P_feas.mean(axis=0)
        std_p_feas  = P_feas.std(axis=0, ddof=0) if P_feas.shape[0] > 1 else np.zeros(P_feas.shape[1])
        mean_s_feas = S_feas.mean(axis=0)
        std_s_feas  = S_feas.std(axis=0, ddof=0) if S_feas.shape[0] > 1 else np.zeros(S_feas.shape[1])
    else:
        mean_p_feas = None
        std_p_feas  = None
        mean_s_feas = None
        std_s_feas  = None

    summary = {
        'mean_p_all': mean_p_all,
        'std_p_all':  std_p_all,
        'mean_p_feasible': mean_p_feas,
        'std_p_feasible':  std_p_feas,
        'mean_s_all': mean_s_all,
        'std_s_all':  std_s_all,
        'mean_s_feasible': mean_s_feas,
        'std_s_feasible':  std_s_feas,
        'results': results,
        'fraction_feasible': feas_mask.mean(),
    }

    # Print summary statistics.
    print("\n================================================")
    print(f"Completed {n_runs} runs (feasible fraction: {summary['fraction_feasible']:.2%})")
    print("Mean(p*) over all runs:")
    print(mean_p_all)
    print("Std(p*)  over all runs:")
    print(std_p_all)
    if mean_p_feas is not None:
        print("\nMean(p*) over feasible runs only:")
        print(mean_p_feas)
        print("Std(p*)  over feasible runs only:")
        print(std_p_feas)

    print("\n---- Microstate entropies s ----")
    print("Mean(s*) over all runs:")
    print(mean_s_all)
    print("Std(s*)  over all runs:")
    print(std_s_all)
    if mean_s_feas is not None:
        print("\nMean(s*) over feasible runs only:")
        print(mean_s_feas)
        print("Std(s*)  over feasible runs only:")
        print(std_s_feas)
    print("================================================\n")

    return summary


## Run the Inference Analysis

Run the full seeded ensemble and save the feasible probability and entropy samples.

In [ ]:
# Sampling and optimization configuration
N_RUNS = 10000
BASE_SEED = 0
MAX_GENERATIONS = 200
DO_POLISH = True
POLISH_MAX_ITERATIONS = 200
POLISH_TIME_LIMIT_SECONDS = 20.0
POLISH_SKIP_IF_GAP_GE = -1e-4
VERBOSE = True
STARTING_DISTRIBUTION = np.mean([cooled_6e7, cooled_2e7], axis=0)

summary = run_batch_fits_safe(
    dist_list,
    dist_high_list,
    dist_low_list,
    tau_list,
    eqm_data,
    n_runs=N_RUNS,
    base_seed=BASE_SEED,
    max_gen=MAX_GENERATIONS,
    do_polish=DO_POLISH,
    polish_maxiter=POLISH_MAX_ITERATIONS,
    polish_time_limit=POLISH_TIME_LIMIT_SECONDS,
    polish_skip_if_gap_ge=POLISH_SKIP_IF_GAP_GE,
    verbose=VERBOSE,
    start_p=STARTING_DISTRIBUTION,
)

# Probability statistics
mean_p = summary['mean_p_all']
std_p = summary['std_p_all']

# Microstate entropy statistics
mean_s = summary['mean_s_all']
std_s = summary['std_s_all']

# Feasible-only statistics
mean_p_feas = summary['mean_p_feasible']
std_p_feas = summary['std_p_feasible']
mean_s_feas = summary['mean_s_feasible']
std_s_feas = summary['std_s_feasible']

# Raw per-run probability and entropy vectors
S = np.vstack([r['s_star'] for r in summary['results']])  # shape: (n_runs, N)
P = np.vstack([r['p_star'] for r in summary['results']])  # shape: (n_runs, N)
feas_mask = np.array([r['feasible'] for r in summary['results']])
S_feas = S[feas_mask]
P_feas = P[feas_mask]

# Save the feasible samples used by the figure notebooks.
np.savetxt('feasible_p.csv', P_feas)
np.savetxt('feasible_s.csv', S_feas)